# 지하철역 - 따릉이 정류장 근접

* **data_processing 노트북 맨 아래에 이어 붙여서 실행** (앞의 DB 연결, `query_df`, `save_df_typed`, `exec_sql`, `df_mismatch` 사용)
* 지도 API는 사용하지 않음 : 지하철역과 정류장 모두 DB에 위도·경도가 있으므로 좌표로 직접 계산
  * 지도 API가 필요한 경우 : 좌표가 없는 정류장을 주소로 변환(지오코딩)하거나, 도보 경로 거리가 필요할 때
* 근처 기준 : 두 지점의 직선거리(haversine) 300m 이내 / 300~500m, 500m 초과는 연결하지 않음
* 사용하는 테이블
  * `raw_subway_station_master` : 서울시 역사마스터 (수집 단계 저장본)
  * `station_both`, `station_diff` : 위 정류장 전처리에서 저장한 정류장
* 만들 테이블 : `station_subway_link` (반경 안의 정류장-지하철역 쌍)

In [ ]:
# ============================================================
# [설정값 + 공통 함수]
# ============================================================
NEAR_RADIUS_M     = 300     # 가까움 구간 경계
MAX_LINK_RADIUS_M = 500     # 연결 최대 반경
SAME_STATION_M    = 700     # 같은 이름의 역이 이 거리 이상 벌어지면 다른 역으로 분리

MANUAL_NAME_MAP = {}        # 역명 표기가 다를 때 수동 매핑 예) {"총신대입구(이수)": "이수"}


def haversine_m(lat1, lon1, lat2, lon2):
    """두 좌표 사이 직선거리(m). numpy 배열 브로드캐스팅 지원"""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 6371008.8 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def normalize_name(name):
    """역명 정규화: 수동 매핑 → 괄호 내용 제거 → 공백 제거 → 끝의 '역' 제거"""
    s = str(name).strip()
    s = MANUAL_NAME_MAP.get(s, s)
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"\s+", "", s)
    return re.sub(r"역$", "", s)


def normalize_line(line):
    """호선 정규화: '01호선', '1호선', '1' → '1호선' / 그 외 노선명은 공백만 제거"""
    s = str(line).strip()
    m = re.fullmatch(r"0*(\d+)\s*(호선)?", s)
    return f"{int(m.group(1))}호선" if m else re.sub(r"\s+", "", s)


def swap_fix(df, lat_col, lon_col):
    """위도 자리에 126~127, 경도 자리에 37 이 들어간 행(뒤바뀜)을 교정. 교정한 행 수 반환"""
    swapped = df[lat_col].between(124, 132) & df[lon_col].between(33, 39)
    df.loc[swapped, [lat_col, lon_col]] = df.loc[swapped, [lon_col, lat_col]].to_numpy()
    return int(swapped.sum())

## 지하철역

In [ ]:
# 값의 형식을 확인해야 전처리 방향을 정할 수 있음
df_sample = query_df("SELECT * FROM raw_subway_station_master LIMIT 10")
display(df_sample)
print(df_sample.dtypes)

In [ ]:
# ============================================================
# [지하철역] raw_subway_station_master에서 필요한 컬럼만 조회
#   ※ 위 샘플의 컬럼명과 다르면 아래 SELECT 의 컬럼명만 맞춰서 수정
# ============================================================
df_subway = query_df("""
    SELECT bldn_id,      -- 역 아이디
           bldn_nm,      -- 역명
           route,        -- 호선
           lat,          -- 위도
           lot           -- 경도
    FROM raw_subway_station_master
""")

print(df_subway.shape)
print(df_subway.dtypes)         # 전부 object(문자열)로 나옴
display(df_subway.head())

In [ ]:
# ============================================================
# [지하철역 전처리] 원본(df_subway)은 그대로 두고 복사본(df_subway2)에서 변환
#   - bldn_id, bldn_nm, route : 문자열(앞뒤 공백 제거)
#   - lat, lot                : 실수(float)
# ============================================================
df_subway2 = df_subway.copy()

# 1) 문자열 컬럼: 문자열 타입으로 바꾸고 앞뒤 공백 제거
for c in ["bldn_id", "bldn_nm", "route"]:
    df_subway2[c] = df_subway2[c].astype("string").str.strip()

# 2) 위도/경도: 실수로 변환. 변환 불가 값은 NaN
df_subway2["lat"] = pd.to_numeric(df_subway2["lat"], errors="coerce").astype("float64")
df_subway2["lot"] = pd.to_numeric(df_subway2["lot"], errors="coerce").astype("float64")

# ---------------- 변환 결과 점검 ----------------
print(df_subway2.dtypes, "\n")
print("결측 수:\n", df_subway2.isna().sum(), "\n")
print("좌표가 0이거나 결측:", int(((df_subway2["lat"] == 0) | (df_subway2["lot"] == 0)
                               | df_subway2["lat"].isna() | df_subway2["lot"].isna()).sum()), "건")
print("bldn_id 중복:", int(df_subway2["bldn_id"].duplicated().sum()), "건")
display(df_subway2.head(10))

In [ ]:
# ============================================================
# [지하철역 좌표 검증 + 물리역] df_subway2 → df_subway_pt(호선별 역) → df_subway_station(물리역)
#   - 위경도가 뒤바뀐 행은 교정, 수도권 범위 밖/결측/0 좌표는 df_subway_err 로 분리
#   - 환승역은 같은 위치의 역 1곳(물리역)으로 묶음
#   - 이름만 같고 SAME_STATION_M 이상 떨어진 역(예: 5호선 양평 / 경의중앙선 양평)은 호선별로 분리
# ============================================================
df_subway3 = df_subway2.copy()
n_swapped = swap_fix(df_subway3, "lat", "lot")

valid = df_subway3["lat"].between(37.0, 38.2) & df_subway3["lot"].between(126.0, 128.0)
df_subway_err = df_subway3[~valid].copy()
df_subway_pt = df_subway3[valid].drop_duplicates("bldn_id").reset_index(drop=True)

df_subway_pt["name_key"] = df_subway_pt["bldn_nm"].map(normalize_name)
df_subway_pt["line_key"] = df_subway_pt["route"].map(normalize_line)

df_subway_pt["phys_key"] = df_subway_pt["name_key"]
for name_key, g in df_subway_pt.groupby("name_key"):
    spread = haversine_m(g["lat"], g["lot"], g["lat"].mean(), g["lot"].mean()).max()
    if spread > SAME_STATION_M:
        df_subway_pt.loc[g.index, "phys_key"] = name_key + "_" + g["line_key"]

df_subway_station = (
    df_subway_pt.groupby("phys_key")
    .agg(subway_name=("bldn_nm", "first"),
         lat=("lat", "mean"), lot=("lot", "mean"),
         lines=("line_key", lambda s: ", ".join(sorted(s.unique()))),
         n_lines=("line_key", "nunique"))
    .reset_index()
)

print(f"원본 {len(df_subway):,}행 → 유효 {len(df_subway_pt):,}행 / 좌표 오류 {len(df_subway_err)}행 / 위경도 뒤바뀜 교정 {n_swapped}행")
print(f"물리역 {len(df_subway_station):,}곳 / 환승역(2개 호선 이상) {(df_subway_station['n_lines'] >= 2).sum():,}곳")
if len(df_subway_err):
    print("\n좌표 오류 역:"); display(df_subway_err.head(20))
display(df_subway_station.head())

## 따릉이 정류장 (위에서 전처리해 저장한 테이블 사용)

* `station_both` : 따릉이·교통정보시스템 양쪽에 있고 값이 동일한 정류장 → 전부 사용
* `station_diff` 중 `data_source = '따릉이'` : 따릉이에만 있는 정류장 → 사용 (대여이력에 나오는 정류장이므로 포함, 구분 컬럼 `station_source`)
* `station_diff` 중 `data_source = '교통정보시스템'` : 이름이 없고 나중에 확인할 정류장 → 제외
* 불일치로 `df_mismatch`에 분리된 정류장은 두 테이블에 없어 제외됨 (처리 기준은 별도로 정할 것)

In [ ]:
# ============================================================
# [정류장] station_both + station_diff(따릉이만) → df_bike_st
# ============================================================
df_st_both = query_df("SELECT rent_id, rent_no, rent_nm, sta_lat, sta_long FROM station_both")
df_st_both["station_source"] = "양쪽일치"

df_st_diff = query_df("""SELECT rent_id, rent_no, rent_nm, sta_lat, sta_long
                         FROM station_diff WHERE data_source = '따릉이'""")
df_st_diff["station_source"] = "따릉이만"

df_bike_st = pd.concat([df_st_both, df_st_diff], ignore_index=True)
df_bike_st["rent_no"]   = pd.to_numeric(df_bike_st["rent_no"], errors="coerce").astype("Int64")
df_bike_st["sta_lat"]   = pd.to_numeric(df_bike_st["sta_lat"], errors="coerce").astype("float64")
df_bike_st["sta_long"]  = pd.to_numeric(df_bike_st["sta_long"], errors="coerce").astype("float64")

# 위경도 뒤바뀜 교정 후, 좌표가 0 / 결측 / 범위 밖인 정류장은 제외
n_swapped = swap_fix(df_bike_st, "sta_lat", "sta_long")
valid = df_bike_st["sta_lat"].between(37.0, 38.2) & df_bike_st["sta_long"].between(126.0, 128.0)
df_bike_st_err = df_bike_st[~valid].copy()
df_bike_st = df_bike_st[valid].reset_index(drop=True)

# ---------------- 점검 ----------------
print(f"station_both {len(df_st_both):,}곳 + station_diff(따릉이만) {len(df_st_diff):,}곳")
print(f"→ 사용 {len(df_bike_st):,}곳 / 좌표 오류로 제외 {len(df_bike_st_err)}곳 / 위경도 뒤바뀜 교정 {n_swapped}곳")
print("rent_id 중복:", int(df_bike_st["rent_id"].duplicated().sum()), "건")
if "df_mismatch" in globals():
    print(f"※ 불일치 정류장(df_mismatch) {len(df_mismatch):,}곳은 포함되지 않았습니다.")
if len(df_bike_st_err):
    print("\n좌표 오류 정류장:"); display(df_bike_st_err.head(20))
display(df_bike_st["station_source"].value_counts().to_frame())

## 근접 관계 계산

In [ ]:
# ============================================================
# [근접 계산] 정류장 × 물리역 직선거리 행렬(numpy) → 반경 500m 안의 쌍만 남김 (df_link)
# ============================================================
dist = haversine_m(
    df_bike_st["sta_lat"].to_numpy()[:, None],
    df_bike_st["sta_long"].to_numpy()[:, None],
    df_subway_station["lat"].to_numpy()[None, :],
    df_subway_station["lot"].to_numpy()[None, :],
)                                                   # shape = (정류장 수, 물리역 수)

bi, si = np.where(dist <= MAX_LINK_RADIUS_M)
df_link = pd.DataFrame({
    "rent_id":        df_bike_st["rent_id"].to_numpy()[bi],
    "rent_no":        df_bike_st["rent_no"].to_numpy()[bi],
    "rent_nm":        df_bike_st["rent_nm"].to_numpy()[bi],
    "station_source": df_bike_st["station_source"].to_numpy()[bi],
    "phys_key":       df_subway_station["phys_key"].to_numpy()[si],
    "subway_name":    df_subway_station["subway_name"].to_numpy()[si],
    "lines":          df_subway_station["lines"].to_numpy()[si],
    "distance_m":     dist[bi, si].round(1),
})
df_link["radius_band"] = np.where(df_link["distance_m"] <= NEAR_RADIUS_M,
                                  f"0-{NEAR_RADIUS_M}m", f"{NEAR_RADIUS_M}-{MAX_LINK_RADIUS_M}m")
# 한 정류장이 여러 역 반경에 걸칠 때 가까운 순서
df_link["subway_rank"] = df_link.groupby("rent_id")["distance_m"].rank(method="first").astype(int)
df_link["is_nearest"]  = df_link["subway_rank"].eq(1)
df_link["radius_m"]    = MAX_LINK_RADIUS_M
df_link = df_link.sort_values(["rent_id", "subway_rank"]).reset_index(drop=True)

# 반경 밖 정류장도 포함한 '가장 가까운 역까지 거리' (확인용)
nearest_m = pd.Series(dist.min(axis=1), index=df_bike_st["rent_id"])

print(f"연결 쌍 {len(df_link):,}건 / 역 근처 정류장 {df_link['rent_id'].nunique():,}곳 "
      f"/ 근처 정류장이 있는 역 {df_link['phys_key'].nunique():,}곳")
display(df_link.head(10))

### 좌표 오류 정류장 → df_diff

* 정류장 불러오기 셀에서 좌표가 0 / 결측 / 범위 밖이라 제외된 정류장(`df_bike_st_err`)을 `df_diff`와 `station_diff` 테이블에 추가해 두고 나중에 정보를 찾음
* `station_both`에서 온 정류장 : 따릉이·교통정보시스템 두 곳에 모두 있는 것을 이미 확인하고 통과한 자료이므로 `data_source`에 두 출처를 모두 기록 (`따릉이,교통정보시스템`)
* `station_diff`에서 온 정류장(따릉이만) : 이미 `df_diff`에 `data_source = 따릉이`로 있으므로 다시 넣지 않음
* 다시 실행해도 중복으로 들어가지 않음

In [ ]:
# ============================================================
# [좌표 오류 정류장 → df_diff] df_bike_st_err 를 df_diff 와 station_diff 테이블에 추가
# ============================================================
BOTH_SOURCE      = "따릉이,교통정보시스템"   # 두 곳 모두에 있는 정류장의 data_source 값
REMOVE_FROM_BOTH = False                    # True: 옮긴 정류장을 station_both 에서도 삭제 / False: 복사만

diff_cols = ["sta_loc", "rent_id", "rent_no", "rent_nm", "sta_addr", "sta_lat", "sta_long", "data_source"]

if "df_bike_st_err" not in globals():
    raise RuntimeError("df_bike_st_err 가 없습니다. 정류장 불러오기 셀을 먼저 실행하세요.")

err_both      = df_bike_st_err[df_bike_st_err["station_source"] == "양쪽일치"]
err_bike_only = df_bike_st_err[df_bike_st_err["station_source"] == "따릉이만"]
ids_both      = err_both["rent_id"].dropna().tolist()

# 1) station_both 에서 온 정류장은 행정구·주소까지 있는 원래 행을 다시 조회 (df_diff 와 같은 컬럼)
if ids_both:
    df_err_add = query_df("""SELECT sta_loc, rent_id, rent_no, rent_nm, sta_addr, sta_lat, sta_long
                            FROM station_both WHERE rent_id = ANY(%s)""", [ids_both])
    for c in ["sta_loc", "rent_id", "rent_nm", "sta_addr"]:
        df_err_add[c] = df_err_add[c].astype("string").str.strip()
    df_err_add["rent_no"]   = pd.to_numeric(df_err_add["rent_no"], errors="coerce").astype("Int64")
    df_err_add["sta_lat"]   = pd.to_numeric(df_err_add["sta_lat"], errors="coerce").astype("float64")
    df_err_add["sta_long"]  = pd.to_numeric(df_err_add["sta_long"], errors="coerce").astype("float64")
    df_err_add["data_source"] = pd.Series(BOTH_SOURCE, index=df_err_add.index, dtype="string")
else:
    df_err_add = pd.DataFrame(columns=diff_cols)

# 2) df_diff(메모리)에 추가 : 이미 들어 있는 rent_id 는 건너뜀
mem_new = df_err_add[~df_err_add["rent_id"].isin(df_diff["rent_id"])]
if len(mem_new):
    df_diff = pd.concat([df_diff, mem_new[diff_cols]], ignore_index=True)
    df_diff["data_source"] = df_diff["data_source"].astype("string")

# 3) station_diff(DB)에 추가 : 이미 저장된 rent_id 는 건너뜀
ensure_conn()
ids_in_db = set(query_df("SELECT rent_id FROM station_diff")["rent_id"])
db_new = df_err_add[~df_err_add["rent_id"].isin(ids_in_db)]
if len(db_new):
    try:
        save_df_typed(db_new[diff_cols], "station_diff", conn, if_exists="append")
    except ValueError as e:
        if "없는 컬럼" in str(e):
            print("※ save_df_typed 의 append 확인 줄이 첫 컬럼만 통과시키는 오류입니다. "
                  "함수를 정의한 셀에서 아래처럼 고친 뒤 그 셀을 다시 실행하고 이 셀을 다시 실행하세요.\n"
                  "   existing = {r[0] for r in cur.fetchall()}\n"
                  "   missing = [c for c in columns if c not in existing]")
        raise

# 4) (선택) station_both 에서도 삭제
if REMOVE_FROM_BOTH and ids_both:
    exec_sql("DELETE FROM station_both WHERE rent_id = ANY(%s)", [ids_both], title="station_both 에서 삭제")
    df_both = df_both[~df_both["rent_id"].isin(ids_both)].reset_index(drop=True)

# ---------------- 결과 점검 ----------------
print(f"좌표 오류 정류장 {len(df_bike_st_err)}곳 = 양쪽일치(station_both) {len(err_both)}곳 + 따릉이만(station_diff) {len(err_bike_only)}곳")
print(f"  양쪽일치 → df_diff 에 새로 추가 {len(mem_new)}곳 / station_diff 에 새로 저장 {len(db_new)}곳 (data_source = {BOTH_SOURCE})")
n_in = int(err_bike_only["rent_id"].isin(df_diff["rent_id"]).sum())
print(f"  따릉이만 → 이미 df_diff 에 있음 {n_in}곳 / 없음 {len(err_bike_only) - n_in}곳")
if len(err_bike_only) - n_in:
    print("  ⚠ df_diff 에 없는 '따릉이만' 정류장이 있습니다. df_diff / station_diff 를 다시 만들었는지 확인하세요.")
print(f"  station_both 에서 삭제: {'함' if REMOVE_FROM_BOTH else '안 함 (REMOVE_FROM_BOTH=False)'}")
display(mem_new if len(mem_new) else df_diff[df_diff["data_source"] == BOTH_SOURCE])
display(query_df("SELECT data_source, COUNT(*) AS cnt FROM station_diff GROUP BY 1 ORDER BY 1"))

In [ ]:
# ============================================================
# [기준 확인] 거리 분포, 반경 밖 정류장, 상식 점검
# ============================================================
for r in (NEAR_RADIUS_M, MAX_LINK_RADIUS_M):
    print(f"가장 가까운 역이 {r}m 이내인 정류장: {(nearest_m <= r).mean():.1%}")
print(f"반경 {MAX_LINK_RADIUS_M}m 안에 역이 없는 정류장: {(nearest_m > MAX_LINK_RADIUS_M).sum():,}곳 / {len(df_bike_st):,}곳")

print("\n가장 가까운 역까지 거리(m) 분포")
display(nearest_m.describe().round(0).to_frame("nearest_m"))
print("역당 근처 정류장 수 분포")
display(df_link.groupby("phys_key").size().describe().round(1).to_frame("정류장 수"))

# 좌표가 한쪽만 틀어진 경우 모든 정류장이 '반경 밖'으로 나옴 → 중심 간 거리로 점검
gap_km = haversine_m(df_bike_st["sta_lat"].mean(), df_bike_st["sta_long"].mean(),
                     df_subway_station["lat"].mean(), df_subway_station["lot"].mean()) / 1000
print(f"\n정류장 중심 ↔ 역 중심 거리: {gap_km:.1f} km  (서울 안이면 수 km 이내가 정상)")
if (nearest_m > MAX_LINK_RADIUS_M).mean() > 0.9 or gap_km > 10:
    print("⚠ 거의 모든 정류장이 반경 밖입니다. 컬럼명(lat/lot)과 좌표 전처리를 다시 확인하세요.")

check_name = "합정"          # 상식 점검 : 아는 역을 넣어 보기
print(f"\n'{check_name}' 근처 정류장 (가까운 순)")
display(df_link[df_link["subway_name"].str.contains(check_name, na=False)].sort_values("distance_m").head(10))

## db 저장

In [ ]:
# ============================================================
# [근접 관계 저장] station_subway_link
#   - save_df_typed 로 타입 자동 지정 후 저장 (거리는 실수, 순위는 정수, is_nearest 는 BOOLEAN)
#   - 기본키는 2개 컬럼이라 저장 후 따로 지정
#   - if_exists="replace" : 계산으로 언제든 다시 만들 수 있는 테이블이라 다시 실행하면 새로 만듦
# ============================================================
link_cols = ["rent_id", "rent_no", "rent_nm", "station_source", "phys_key", "subway_name", "lines",
             "distance_m", "radius_band", "subway_rank", "is_nearest", "radius_m"]

ensure_conn()
save_df_typed(df_link[link_cols], "station_subway_link", conn, if_exists="replace")
exec_sql("ALTER TABLE station_subway_link ADD PRIMARY KEY (rent_id, phys_key)", title="기본키 지정")
exec_sql("CREATE INDEX IF NOT EXISTS idx_station_subway_link_phys ON station_subway_link (phys_key)", title="인덱스 생성")

# 저장된 컬럼 타입 확인
display(query_df("""SELECT column_name, data_type FROM information_schema.columns
                    WHERE table_schema = current_schema() AND table_name = 'station_subway_link'
                    ORDER BY ordinal_position"""))

In [ ]:
# ============================================================
# [저장 결과 확인]
# ============================================================
chk = query_df("""SELECT COUNT(*) AS n, COUNT(DISTINCT rent_id) AS rents, COUNT(DISTINCT phys_key) AS subways,
                           SUM(is_nearest::int) AS nearest FROM station_subway_link""")
print(chk.to_string(index=False))
print("DataFrame 과 행 수:", "일치 ✔" if int(chk["n"][0]) == len(df_link) else "불일치 ⚠")

display(query_df("""SELECT radius_band, COUNT(*) AS cnt, ROUND(AVG(distance_m)::numeric, 1) AS avg_m
                    FROM station_subway_link GROUP BY 1 ORDER BY 1"""))
display(query_df("SELECT * FROM station_subway_link ORDER BY rent_id, subway_rank LIMIT 5"))